## Multi-Tool Calling with MAF and Microsoft Foundry

![multiple-tools](./Assets/multiple-tools.png)

### Installing Dependencies and Libraries

In [ ]:
%pip install agent-framework-core agent-framework-foundry python-dotenv azure-ai-projects


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)


Project Endpoint:  https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
Model:  gpt-5.2


### Loading the Code Interpreter Tool

In [10]:
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Foundry chat client - hosted tools are created from this client
chat_client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model,
    credential=credential,
)

# Hosted code interpreter tool (runs server-side in Foundry)
code_interpreter_tool = chat_client.get_code_interpreter_tool()

### Loading the Foundry Hosted MCP Tool

In [ ]:
# Hosted MCP tool pointing at the Microsoft Learn MCP server
ms_learn_mcp_tool = chat_client.get_mcp_tool(
    name="Microsoft Learn MCP Tool",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="never_require",
)

### Creating our Multi-Tooled Agent

In [12]:
from agent_framework import Agent

# Multi-tooled agent using the Foundry client and both hosted tools
agent = Agent(
    name="multi-tool-agent",
    instructions="You are a helpful multi-tool agent",
    client=chat_client,
    tools=[code_interpreter_tool, ms_learn_mcp_tool],
)

session = agent.create_session()


### Running Different Queries on the Agent

In [13]:
query = "Can you write a Python function to calculate modulus of two numbers and give me the output for numbers 10 and 3"
response = await agent.run(query, session=session)

print("\n🧠 Assistant:\n", response)



🧠 Assistant:
 ```python
def modulus(a, b):
    return a % b

# Example
result = modulus(10, 3)
print(result)
```

Output:
```
1
```


In [9]:
query = "How to define azure default credentials in ai-103 certification path?"
response = await agent.run(query, session=session)

print("\n🧠 Assistant:\n", response)



🧠 Assistant:
 In the AI-103/Azure AI Engineer context, “Azure default credentials” usually refers to using **`DefaultAzureCredential`** from the **Azure Identity** SDK. It’s the recommended way to authenticate to Azure services (including Azure AI services) without hard-coding secrets, because it automatically tries multiple auth sources depending on where your code runs (local dev vs Azure).

## Python: define and use DefaultAzureCredential
1) Install:
```bash
pip install azure-identity
```

2) In code:
```python
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
```

That `credential` object is then passed into Azure SDK clients that support Microsoft Entra ID authentication.

## How DefaultAzureCredential decides what to use (Python)
When a token is needed, `DefaultAzureCredential` tries these in order until one works (high level): **environment-based service principal**, **workload identity**, **managed identity**, then developer sign-in sourc

### Cleaning Up

Close the agent to release its client connection and resources.


In [15]:
await agent.close()
credential.close()
